# Chapter 08 — Polars and Apache Arrow for Efficient Data Processing

Polars offers an expression-based DataFrame API and lazy query optimization. Apache Arrow is a columnar memory format for efficient data interchange.

## Prerequisites
Python, Pandas, SQL-style filtering/grouping; install `polars` and `pyarrow` for full examples.

## Learning goals
- Explain eager versus lazy execution and query plans.
- Use Polars expressions to select, filter, add, and aggregate columns.
- Read files lazily and collect results deliberately.
- Understand Arrow tables and interoperability.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Guided learning path

Work from top to bottom. Each step states the purpose, explains the API choice, and provides a complete code cell. Modify parameters and predict the output before running it.

### Step 1 — Guided 1 — Compare eager and lazy

Eager mode materializes an object immediately. Lazy mode builds a plan until `collect()` executes it.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
try:
 import polars as pl
 path=DATA/"sales.csv"
 eager=pl.read_csv(path); lazy=pl.scan_csv(path)
 print("eager:",eager.shape); print(lazy.explain())
except ImportError: print("pip install polars")

Data directory: /mnt/data/python_data_analysis_book/data
pip install polars


### Step 2 — Guided 2 — Filter and derive a column

Use `pl.col` to refer to a column and `.alias` to name an expression result.

In [3]:
try:
 filtered=eager.with_columns((pl.col("revenue")-pl.col("cost")).alias("profit_check")).filter(pl.col("region")=="East")
 display(filtered.head())
except NameError: print("Run Guided 1 after installing Polars")

Run Guided 1 after installing Polars


### Step 3 — Guided 3 — Aggregate lazily

A lazy result is not yet materialized; `collect` executes the plan.

In [4]:
try:
 query=lazy.group_by("region").agg(pl.col("revenue").sum().alias("revenue_total"),pl.len().alias("orders")).sort("revenue_total",descending=True)
 display(query.collect())
except NameError: print("Run Guided 1 after installing Polars")

Run Guided 1 after installing Polars


### Step 4 — Guided 4 — Inspect Arrow schema

Arrow tables carry column types and metadata. Their schema is important for interoperability and downstream type handling.

In [5]:
try:
 table=eager.to_arrow(); print(table.schema)
except Exception as e: print("Optional Arrow setup:",e)

Optional Arrow setup: name 'eager' is not defined


## Tutorial checkpoint

1. What was the input data type?
2. What did the function or method return?
3. Which argument most changed the result?
4. What edge case should be tested before using this in production?